# 🐝 TTC: Insects Standard SupCon (50:50 & 99:1) [Table 1]
Tự động chuẩn bị Dataset iNat21, Pretrain 350 Epochs, Linear Probe 50 Epochs, lưu Checkpoint vào `/kaggle/working/` và log WandB.

In [ ]:
# 1. Kiểm tra GPU
!nvidia-smi

In [ ]:
# 2. Clone mã nguồn TTC
import os
if not os.path.exists('TTC'):
    !git clone https://github.com/XiaoSha59/TTC.git
%cd TTC
!git pull

In [ ]:
# 3. Cài đặt các thư viện
!pip install -q 'lightning>=2.0.0' 'hydra-core>=1.3.2' omegaconf pyrootutils timm

In [ ]:
# 4. Cấu hình WandB (Tùy chọn: có thể thay bằng key riêng nếu muốn)
import os, wandb
os.environ['WANDB_API_KEY'] = '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'
!wandb login $WANDB_API_KEY

In [ ]:
# 5. Thiết lập Dataset iNat21 Natural (Tự động nhận diện hoặc tải nhanh từ AWS S3)
import os, glob, subprocess, shutil
os.makedirs('data/inat21', exist_ok=True)

# Kiểm tra train_mini trong input
train_cands = glob.glob('/kaggle/input/**/train_mini', recursive=True)
if train_cands:
    train_src = train_cands[0]
    print(f'>>> Tìm thấy train_mini có sẵn: {train_src}')
else:
    print('>>> Đang tải nhanh train_mini từ AWS S3...')
    !wget -q --show-progress -O /tmp/train_mini.tar.gz https://ml-inat-competition-datasets.s3.amazonaws.com/2021/train_mini.tar.gz
    !tar -xzf /tmp/train_mini.tar.gz -C /tmp/
    !rm -f /tmp/train_mini.tar.gz
    train_src = '/tmp/train_mini'

if not os.path.exists('data/inat21/train_mini'):
    os.symlink(train_src, 'data/inat21/train_mini')
if not os.path.exists('data/inat21/train'):
    os.symlink(train_src, 'data/inat21/train')

# Tải tập val chuẩn (8.3GB)
if not os.path.exists('/tmp/val'):
    print('>>> Đang tải tập val từ AWS S3...')
    !wget -q --show-progress -O /tmp/val.tar.gz https://ml-inat-competition-datasets.s3.amazonaws.com/2021/val.tar.gz
    print('>>> Đang giải nén tập val vào /tmp...')
    !tar -xzf /tmp/val.tar.gz -C /tmp/
    !rm -f /tmp/val.tar.gz

if not os.path.exists('data/inat21/val'):
    os.symlink('/tmp/val', 'data/inat21/val')

# Kiểm tra số lượng ảnh
from data.iNatData import INaturalistNClasses
t_ds = INaturalistNClasses('data/inat21', split='train', classes=['Animalia_Arthropoda_Insecta_Hymenoptera_Apidae', 'Animalia_Arthropoda_Insecta_Hymenoptera_Vespidae'])
v_ds = INaturalistNClasses('data/inat21', split='val', classes=['Animalia_Arthropoda_Insecta_Hymenoptera_Apidae', 'Animalia_Arthropoda_Insecta_Hymenoptera_Vespidae'])
print(f'✅ Dataset iNat21 sẵn sàng! Train: {len(t_ds)} ảnh, Val: {len(v_ds)} ảnh')

In [ ]:
# 6. [MODEL 1/2] Pre-training 350 Epochs Standard SupCon 50:50
print('🚀 [1/4] PRE-TRAIN 350 EPOCHS STANDARD SUPCON 50:50...')
!python train.py \
    experiment=contrastive \
    experiment/specs=insects \
    class_ratios=[0.5,0.5] \
    module.ratio_supervised_majority=1.0 \
    batch_size=256 \
    trainer.max_epochs=350 \
    module.lr=0.0625 \
    trainer.precision=16-mixed \
    data.data_module.num_workers=2 \
    data.data_module.persistent_workers=False \
    trainer.check_val_every_n_epoch=5 \
    name='insects-50_50-supcon-350ep-full'

In [ ]:
# 7. [MODEL 1/2] Linear Probing 50:50 Standard SupCon & Lưu Checkpoint
import glob, os, shutil
ckpts = sorted(glob.glob('logs/train/runs/*/checkpoints/last.ckpt'), key=os.path.getmtime)
last_ckpt = ckpts[-1]
os.makedirs('/kaggle/working/saved_checkpoints', exist_ok=True)
saved_50_50 = '/kaggle/working/saved_checkpoints/insects_50_50_supcon_350ep_backbone.ckpt'
shutil.copyfile(last_ckpt, saved_50_50)
print(f'✅ Checkpoint 50:50 SupCon: {saved_50_50}')

print('🚀 [2/4] LINEAR PROBING 50:50 STANDARD SUPCON...')
!python train.py \
    experiment=finetune \
    experiment/specs=insects \
    +base_model_path={saved_50_50} \
    trainer.max_epochs=50 \
    module.optimizer_name=adam \
    module.lr=0.001 \
    train_transform._target_=data.augmentation.SimCLRValTransform \
    data.data_module.num_workers=2 \
    data.data_module.persistent_workers=False \
    name='insects-50_50-supcon-official-probe'

In [ ]:
# 8. [MODEL 2/2] Pre-training 350 Epochs Standard SupCon 99:1
print('🚀 [3/4] PRE-TRAIN 350 EPOCHS STANDARD SUPCON 99:1...')
!python train.py \
    experiment=contrastive \
    experiment/specs=insects \
    class_ratios=[0.01,0.99] \
    module.ratio_supervised_majority=1.0 \
    batch_size=256 \
    trainer.max_epochs=350 \
    module.lr=0.0625 \
    trainer.precision=16-mixed \
    data.data_module.num_workers=2 \
    data.data_module.persistent_workers=False \
    trainer.check_val_every_n_epoch=5 \
    name='insects-99_1-supcon-350ep-full'

In [ ]:
# 9. [MODEL 2/2] Linear Probing 99:1 Standard SupCon & Lưu Checkpoint
import glob, os, shutil
ckpts = sorted(glob.glob('logs/train/runs/*/checkpoints/last.ckpt'), key=os.path.getmtime)
last_ckpt = ckpts[-1]
saved_99_1 = '/kaggle/working/saved_checkpoints/insects_99_1_supcon_350ep_backbone.ckpt'
shutil.copyfile(last_ckpt, saved_99_1)
print(f'✅ Checkpoint 99:1 SupCon: {saved_99_1}')

print('🚀 [4/4] LINEAR PROBING 99:1 STANDARD SUPCON...')
!python train.py \
    experiment=finetune \
    experiment/specs=insects \
    +base_model_path={saved_99_1} \
    trainer.max_epochs=50 \
    module.optimizer_name=adam \
    module.lr=0.001 \
    train_transform._target_=data.augmentation.SimCLRValTransform \
    data.data_module.num_workers=2 \
    data.data_module.persistent_workers=False \
    name='insects-99_1-supcon-official-probe'

In [ ]:
# 10. Tổng kết
print('🎉🎉🎉 HOÀN THÀNH XUẤT SẮC CẢ 2 MODELS STANDARD SUPCON (50:50 & 99:1)!')
!ls -lh /kaggle/working/saved_checkpoints